# CP2 LastFM — Analisis Gap dan Persiapan Time Embedding

Notebook Colab tahap berikutnya. Ia menerima file hasil audit dengan upload langsung, memeriksa gap dan sequence, merangkum distribusi, lalu membangun versi **bucket waktu untuk inspeksi**. Notebook ini belum menjalankan model atau training dan tidak menimpa file CP1.

Catatan penting: HetRec LastFM timestamp mengukur waktu pemberian tag artis, bukan waktu mendengarkan/scrobble. Jadi eksperimen menguji sinyal waktu tagging.

Sebelum memakai bucket dalam model, cocokkan split dan DataLoader ini dengan source BSARec yang dipakai pada CP1. Jangan membandingkan hasil training sampai baseline dan varian memakai persis split, seed, training budget, serta evaluator yang sama.


## 1. Upload file hasil audit


In [ ]:
from google.colab import files
from pathlib import Path
uploaded=files.upload()
ITEM_FILE=Path("/content/LastFM.txt")
TIME_FILE=Path("/content/LastFM_time.txt")
assert ITEM_FILE.exists() and TIME_FILE.exists(), "Upload LastFM.txt dan LastFM_time.txt"


## 2. Baca sequence dan validasi alignment


In [ ]:
from collections import OrderedDict, Counter
import json, math, statistics, hashlib

def read_seq(path):
    out=OrderedDict()
    for no,line in enumerate(path.read_text(encoding="utf-8-sig").splitlines(),1):
        p=list(map(int,line.split()))
        assert len(p)>=2, f"Baris kosong/pendek {no}"
        assert p[0] not in out, f"User ID berulang {p[0]}"
        out[p[0]]=p[1:]
    return out
items=read_seq(ITEM_FILE); gaps=read_seq(TIME_FILE)
assert list(items)==list(gaps), "Urutan/user ID berbeda"
assert all(len(items[u])==len(gaps[u]) for u in items), "Panjang item dan gap tidak sama"
assert all(g[0]==0 and min(g)>=0 for g in gaps.values()), "Gap pertama harus 0 dan tidak boleh negatif"
assert all(len(s)==len(set(s)) for s in items.values()), "Masih ada item berulang dalam sequence"
print("PASS: users",len(items),"items",len({i for s in items.values() for i in s}),"events",sum(map(len,items.values())))


## 3. Analisis distribusi gap

Zero gap dipisahkan karena pada data tag bisa berarti timestamp sama, bukan tidak ada waktu berlalu secara perilaku dengar. Tiga gap panjang ditinjau; jangan otomatis dibuang hanya karena ekstrem.


In [ ]:
gap_records=[(u,pos,g) for u,seq in gaps.items() for pos,g in enumerate(seq) if pos>0]
vals=[g for _,_,g in gap_records]
pos=[g for g in vals if g>0]
large=[(u,p,g,round(g/86400,2)) for u,p,g in gap_records if g>10*365*86400]
print("Adjacent gaps:",len(vals),"zero:",sum(g==0 for g in vals),"positive:",len(pos))
if pos:
    print("Positive gap seconds median/P90/P99/max:",statistics.median(pos),statistics.quantiles(pos,n=10)[8],sorted(pos)[int(.99*(len(pos)-1))],max(pos))
print("Gaps > 10 years:",large)
print("Warning: high gaps stay in data; their source semantics require review.")


## 4. Tetapkan bucket waktu yang transparan

Untuk analisis awal, gunakan bucket log2 dalam satuan jam: bucket 0 khusus gap nol; bucket positif bertambah setiap kali besaran gap kira-kira berlipat dua. Ini meredam rentang ekstrem tanpa menghapus data. Simpan definisi dan hitungan bucket agar dapat direproduksi.


In [ ]:
HOUR=3600

def gap_bucket(g):
    if g==0: return 0
    # Bucket 1: (0, 1h]; 2: (1h, 2h]; 3: (2h, 4h] ...
    return 1 + int(math.ceil(math.log2(g/HOUR))) if g>HOUR else 1

bucket_seqs=OrderedDict((u,[gap_bucket(g) for g in seq]) for u,seq in gaps.items())
bucket_counts=Counter(b for seq in bucket_seqs.values() for b in seq)
print("Bucket counts:",dict(sorted(bucket_counts.items())))
print("Unique buckets:",len(bucket_counts),"max bucket:",max(bucket_counts))



## 5. Cek pergeseran histori-target agar tidak bocor

Untuk sequence item x[0..n-1], target berikutnya pada posisi t menerima histori item sampai t-1. Gap/bucket untuk item histori itu berada pada posisi yang sama. Bucket di posisi target tidak dimasukkan ke input. Contoh berikut membuat satu sample dengan panjang maksimum 50 untuk diperiksa secara manual.


In [ ]:
MAX_LEN=50
uid=next(iter(items))
seq=items[uid]; bseq=bucket_seqs[uid]
# Target paling akhir seperti protokol test next-item umum; input tidak mencakup item target.
target=seq[-1]
hist_items=seq[:-1][-MAX_LEN:]
hist_buckets=bseq[:-1][-MAX_LEN:]
assert len(hist_items)==len(hist_buckets)
assert target not in hist_items or True  # item boleh muncul kembali lintas histori pada data lain
print("sample uid",uid,"history length",len(hist_items),"target item",target)
print("history items[:10]",hist_items[:10])
print("history time buckets[:10]",hist_buckets[:10])
print("target bucket excluded; full target-position bucket was",bseq[-1])


## 6. Simpan file bucket dan laporan

Output tetap merupakan artefak persiapan; belum menjadi input model sampai DataLoader CP1 disesuaikan secara terkendali.


In [ ]:
OUT=Path("/content/cp2_time_prep")
OUT.mkdir(exist_ok=True)
bucket_file=OUT/"LastFM_cp2_gap_buckets.txt"
with bucket_file.open("w",encoding="utf-8",newline="\n") as f:
    for uid,seq in bucket_seqs.items(): f.write(str(uid)+" "+" ".join(map(str,seq))+"\n")
report={
 "users":len(items),"artists":len({i for s in items.values() for i in s}),"interactions":sum(map(len,items.values())),
 "adjacent_gap_count":len(vals),"zero_gap_count":sum(g==0 for g in vals),"positive_gap_count":len(pos),
 "gaps_over_10_years":large,"bucket_definition":"0 for zero; 1 for 0<gap<=1h; log2-hour buckets above 1h",
 "bucket_counts":dict(sorted(bucket_counts.items())),
 "input_target_rule":"target timestamp/bucket excluded; only history positions supplied",
 "source_semantics":"HetRec LastFM tag time, not listening/scrobble time",
 "source_hashes":{"items":hashlib.sha256(ITEM_FILE.read_bytes()).hexdigest(),"gaps":hashlib.sha256(TIME_FILE.read_bytes()).hexdigest()}
}
report_file=OUT/"CP2_time_feature_prep_report.json"
report_file.write_text(json.dumps(report,indent=2),encoding="utf-8")
print(json.dumps(report,indent=2))
files.download(str(bucket_file))
files.download(str(report_file))


## Gerbang sebelum implementasi model

1. Konfirmasi file bucket dan definisi gap diterima sebagai satu eksperimen eksploratif.
2. Cocokkan split validasi/test dengan loader BSARec CP1. Jangan memakai split buatan notebook ini untuk klaim komparatif.
3. Integrasikan embedding bucket ke jalur embedding item pada salinan kode CP2, dengan mask histori dan posisi target diperiksa.
4. Jalankan baseline CP1 dan BSARec+time pada seed dan protokol identik. Pertahankan objective BSARec.
5. Laporkan delta HR/NDCG@5/10/20 per seed, mean, dan SD, sambil menyebut timestamp sebagai waktu tag.

Notebook ini belum melatih model. Source loader/model BSARec CP1 harus tersedia sebelum integrasi agar baseline tidak berubah.
